In [ ]:
# 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import make_scorer, mean_squared_error

# Fungsi metric: Root Mean Squared Error (RMSE)
# Ini adalah metrik yang dipakai di kompetisi Kaggle (RMSLE)
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# Scorer untuk cross-validation
rmse_scorer = make_scorer(rmse, greater_is_better=False)

In [ ]:
# 2. Load Data yang Sudah Diproses

# Kita muat data bersih dari folder `output/` (hasil dari notebook feature_engineering).

In [ ]:
train_df = pd.read_csv('output/train_processed.csv')
test_df = pd.read_csv('output/test_processed.csv')
sample_sub = pd.read_csv('data/sample_submission.csv')

print(f"Bentuk data train bersih: {train_df.shape}")
print(f"Bentuk data test bersih: {test_df.shape}")

In [ ]:
# 3. Siapkan Data untuk Model

# Pisahkan kembali fitur (X) dan target (y).

In [ ]:
# Target kita adalah 'SalePrice_Log' yang sudah kita buat di notebook sebelumnya
y_train = train_df['SalePrice_Log']

# Fitur kita adalah sisa kolomnya
X_train = train_df.drop('SalePrice_Log', axis=1)

X_test = test_df.copy()

# PENTING: Pastikan urutan kolom di X_train dan X_test sama persis
# Ini untuk jaga-jaga jika ada kolom yg beda setelah di-encode
X_train, X_test = X_train.align(X_test, join='inner', axis=1, fill_value=0)

print(f"Jumlah fitur yang akan dipakai: {X_train.shape[1]}")

In [ ]:
# 4. Cross-Validation Model

# Kita uji performa model (LGBM) dengan 5-Fold Cross-Validation. Ini untuk melihat seberapa bagus model kita sebelum di-submit.

In [ ]:
# Kita pakai LightGBM, model yg cepat dan akurat
model_lgb = lgb.LGBMRegressor(objective='regression',
                              num_leaves=5,
                              learning_rate=0.05,
                              n_estimators=720,
                              random_state=42)

# Siapkan KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Jalankan cross-validation
# Kita pakai -rmse_scorer karena scikit-learn memaksimalkan, jadi kita maksimalkan -RMSE (alias minimalkan RMSE)
cv_scores = cross_val_score(model_lgb, X_train.values, y_train.values, 
                            scoring=rmse_scorer, cv=kf)

print("--- Hasil Cross-Validation ---")
print(f"Skor RMSE (Log Price) per fold: {-cv_scores}")
print(f"Rata-rata RMSE (Log Price): {-cv_scores.mean():.5f}")

In [ ]:
# 5. Latih Model Final dan Prediksi

# Setelah yakin modelnya bagus, kita latih model pada SEMUA data train dan buat prediksi pada data test.

In [ ]:
print("Melatih model final pada semua data train...")
model_lgb.fit(X_train, y_train)

# Buat prediksi (hasilnya masih dalam Log Price)
predictions_log = model_lgb.predict(X_test)

# Kembalikan ke nilai aslinya (Anti-Log)
# Kita pakai np.expm1 karena kita pakai np.log1p di awal
predictions = np.expm1(predictions_log)

print("Prediksi selesai.")

In [ ]:
# 6. Buat File Submission

# Simpan hasil prediksi ke `submission.csv` sesuai format Kaggle.

In [ ]:
submission_df = pd.DataFrame()
submission_df['Id'] = sample_sub['Id']
submission_df['SalePrice'] = predictions

# Simpan ke folder output
submission_df.to_csv("output/submission.csv", index=False)

print("File 'submission.csv' berhasil disimpan di folder 'output/'.")
print("Siap untuk di-submit ke Kaggle!")

# Tampilkan 5 baris pertama dari hasil prediksi
display(submission_df.head())